# Evaluation Metrics

Accuracy is the first number everyone reports and often the least informative. This notebook shows:

1. The **accuracy paradox** on imbalanced data
2. **Confusion matrix**, **precision**, **recall**, **F1**, computed by hand and checked against scikit-learn
3. **Threshold tuning**, **ROC** and **precision-recall** curves
4. Fixing imbalance with `pos_weight`
5. Multi-class evaluation: per-class report, macro vs weighted averages

Everything runs in seconds on CPU.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics as skm

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Imbalanced data: 95% negative, 5% positive

Think fraud detection or rare disease screening. A model that always predicts "negative" scores about 95% accuracy and catches nothing.

In [ ]:
X, y = make_classification(n_samples=6000, n_features=10, n_informative=5,
                           weights=[0.95, 0.05], class_sep=1.0, flip_y=0.01,
                           random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_tr)           # fit on train only
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)

X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
y_tr, y_te = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1), torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)

print(f'positives: train {int(y_tr.sum())}/{len(y_tr)}, test {int(y_te.sum())}/{len(y_te)}')
print(f'"always negative" accuracy on test: {1 - y_te.mean().item():.3f}')

## 2. Train a small classifier

In [ ]:
def train_binary(pos_weight=None, epochs=200):
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
    pw = None if pos_weight is None else torch.tensor([pos_weight])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        criterion(model(X_tr), y_tr).backward()
        opt.step()
    model.eval()
    return model

@torch.no_grad()
def predict_proba(model, X):
    return torch.sigmoid(model(X)).squeeze(1).numpy()

model = train_binary()
probs = predict_proba(model, X_te)
y_true = y_te.squeeze(1).numpy().astype(int)
y_pred = (probs >= 0.5).astype(int)
print(f'accuracy: {(y_pred == y_true).mean():.3f}')

The accuracy looks great, which says nothing yet. The confusion matrix shows where the model is wrong.

## 3. Confusion matrix, precision, recall, F1 by hand

|  | predicted 0 | predicted 1 |
|---|---|---|
| **actual 0** | TN | FP |
| **actual 1** | FN | TP |

- **Precision** = TP / (TP + FP): of everything flagged positive, how much was right?
- **Recall** = TP / (TP + FN): of all real positives, how many did we catch?
- **F1** = harmonic mean of the two; low if either is low.

In [ ]:
def confusion(y_true, y_pred):
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return tn, fp, fn, tp

def prf(tn, fp, fn, tp):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall    = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

tn, fp, fn, tp = confusion(y_true, y_pred)
p, r, f1 = prf(tn, fp, fn, tp)
print(f'TN={tn} FP={fp} FN={fn} TP={tp}')
print(f'precision={p:.3f} recall={r:.3f} F1={f1:.3f}')

# check against scikit-learn
assert np.allclose([p, r, f1], [skm.precision_score(y_true, y_pred, zero_division=0),
                                skm.recall_score(y_true, y_pred),
                                skm.f1_score(y_true, y_pred)])
assert (np.array([[tn, fp], [fn, tp]]) == skm.confusion_matrix(y_true, y_pred)).all()
print('matches scikit-learn')

## 4. The threshold is a dial

The model outputs a probability. Predicting positive at `p >= 0.5` is a default, not a law. Lowering the threshold catches more positives (recall up) at the cost of more false alarms (precision down).

In [ ]:
rows = []
for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    pred = (probs >= t).astype(int)
    p, r, f1 = prf(*confusion(y_true, pred))
    rows.append((t, p, r, f1))
print(f"{'threshold':>9}{'precision':>11}{'recall':>9}{'F1':>7}")
for t, p, r, f1 in rows:
    print(f'{t:>9.1f}{p:>11.3f}{r:>9.3f}{f1:>7.3f}')

## 5. ROC and precision-recall curves

Sweeping every threshold gives a curve instead of a single point.

- **ROC** plots recall (TPR) against false-positive rate. AUC is the chance a random positive ranks above a random negative.
- **Precision-recall** plots precision against recall. On heavily imbalanced data it is more honest than ROC, because ROC's false-positive rate is diluted by the many negatives.

In [ ]:
fpr, tpr, _ = skm.roc_curve(y_true, probs)
prec, rec, _ = skm.precision_recall_curve(y_true, probs)
roc_auc = skm.roc_auc_score(y_true, probs)
ap = skm.average_precision_score(y_true, probs)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax[0].set(xlabel='false positive rate', ylabel='recall (TPR)', title=f'ROC  AUC={roc_auc:.3f}')
ax[1].plot(rec, prec); ax[1].axhline(y_true.mean(), color='k', ls='--', alpha=0.4, label='random')
ax[1].set(xlabel='recall', ylabel='precision', title=f'Precision-recall  AP={ap:.3f}'); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Training for imbalance: `pos_weight`

`BCEWithLogitsLoss(pos_weight=w)` multiplies the loss of positive samples by `w`, so missing a rare positive hurts more. A common start is `negatives / positives`.

In [ ]:
w = float((y_tr == 0).sum() / (y_tr == 1).sum())
print(f'pos_weight = {w:.1f}')
model_w = train_binary(pos_weight=w)
probs_w = predict_proba(model_w, X_te)

print(f"{'model':<12}{'acc':>7}{'precision':>11}{'recall':>9}{'F1':>7}{'ROC AUC':>9}{'AP':>7}")
for name, pr in [('plain', probs), ('pos_weight', probs_w)]:
    pred = (pr >= 0.5).astype(int)
    p, r, f1 = prf(*confusion(y_true, pred))
    print(f'{name:<12}{(pred == y_true).mean():>7.3f}{p:>11.3f}{r:>9.3f}{f1:>7.3f}'
          f'{skm.roc_auc_score(y_true, pr):>9.3f}{skm.average_precision_score(y_true, pr):>7.3f}')

## 7. Multi-class: the 8x8 digits dataset

With more than two classes, precision/recall/F1 are computed **per class**, then averaged:

- **macro**: plain mean over classes, so every class counts equally (exposes weak rare classes)
- **weighted**: mean weighted by class size (closer to accuracy)

In [ ]:
digits = load_digits()
Xd = StandardScaler().fit_transform(digits.data)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, digits.target, test_size=0.3, stratify=digits.target, random_state=SEED)
Xd_tr, Xd_te = torch.tensor(Xd_tr, dtype=torch.float32), torch.tensor(Xd_te, dtype=torch.float32)
yd_tr_t = torch.tensor(yd_tr)

torch.manual_seed(SEED)
clf = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 10))
opt = torch.optim.Adam(clf.parameters(), lr=1e-2)
for _ in range(150):
    clf.train(); opt.zero_grad()
    nn.functional.cross_entropy(clf(Xd_tr), yd_tr_t).backward(); opt.step()

clf.eval()
with torch.no_grad():
    pred_d = clf(Xd_te).argmax(1).numpy()

print(skm.classification_report(yd_te, pred_d, digits=3))

In [ ]:
cm = skm.confusion_matrix(yd_te, pred_d)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set(xlabel='predicted', ylabel='actual', title='Digits confusion matrix', xticks=range(10), yticks=range(10))
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 8. Which metric to use

| Situation | Look at |
|---|---|
| Balanced classes, equal error costs | accuracy |
| Missing a positive is costly (disease, fraud) | recall, then tune the threshold |
| False alarms are costly (spam filter, alerts) | precision |
| Need one number for imbalanced data | F1 or average precision |
| Comparing models independent of threshold | ROC AUC (balanced) or AP (imbalanced) |
| Multi-class with rare classes | macro F1 plus the per-class report |

Pick the threshold on **validation** data, then report final numbers once on test data.